In [1]:
# Colab only: clone the repo and move into the project root
import os
import sys

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    # Clone repository if it hasn't been cloned yet
    if not os.path.exists('/content/convolutional_vae_quishing'):
        !git clone https://github.com/AnhJoe/convolutional_vae_quishing.git

    # Change working directory to the project root
    os.chdir('/content/convolutional_vae_quishing')

In [2]:
#| output: false
# Local only: find the project root (the folder containing src/) and add it to the import path
# For local runs. Else if running Colab, skip this.

import sys
from pathlib import Path

# Set to root for src load
root = Path.cwd()
while root != root.parent and not (root / "src").exists():
    root = root.parent

if not (root / "src").exists():
    raise FileNotFoundError("Could not find 'src' folder above current working directory.")

sys.path.insert(0, str(root))
print("Project root:", root)

Project root: c:\Users\joetn\convolutional_vae_quishing


# Data processing

This notebook turns the cleaned manifest from `1_eda.ipynb` into one shared table, the **items table**, that the model notebooks read. Nothing here trains a model. It is run once, and the model notebooks (`3_floors.ipynb` onward) only read its output, so everyone trains, tunes and tests on identical data.

Run `1_eda.ipynb` first. It downloads the dataset into `data/raw` and saves `data/processed/manifest.csv.gz`, the list of every URL with its cleaning decision. This notebook then does two things:

1. Assigns every URL to train, validation or test, using the grouped rule decided in `1_eda.ipynb` section 5, and saves the items table.
2. Checks the splits.

Everything is saved in `data/processed/`, which is git-ignored. The column-by-column description of the items table is at the top of `src/prepare_data.py`.

## 1. Load the cleaned manifest

The manifest has one row per URL from the source CSVs. Its `keep` column is false only for the 38 rows whose URL appears in both classes (`1_eda.ipynb` section 6), so the items table has 1,005,700 rows.

In [3]:
# Imports and paths
import numpy as np
import pandas as pd

from src import prepare_data
from src.splits import SPLIT_NAMES

PROC_DIR = root / "data" / "processed"

if not (PROC_DIR / "manifest.csv.gz").exists():
    raise FileNotFoundError("data/processed/manifest.csv.gz not found. Run 1_eda.ipynb first.")
manifest = pd.read_csv(PROC_DIR / "manifest.csv.gz", usecols=["cls", "keep", "drop_reason"], keep_default_na=False)
print(f"{len(manifest):,} rows in the manifest, {int(manifest['keep'].sum()):,} kept")
manifest.groupby("cls")["keep"].agg(rows="size", kept="sum")

1,005,738 rows in the manifest, 1,005,700 kept


,rows,kept
cls,,
benign,429976,429957
malicious,575762,575743


## 2. Build the items table

One row per kept URL. The table joins the manifest (URL, label and URL features) and the split, and saves them to `items.csv.gz`.

**The split rule** was decided in `1_eda.ipynb` section 5 and is applied here without change:

- Benign and malicious rows are each split 70/15/15 into train, validation and test (70% for training, 15% for tuning, 15% for the final test).
- The split is grouped by domain (the site name, for example `wikipedia.org`), so all rows of one domain stay in one split. This stops a model from scoring well just because it saw other pages of the same site during training.
- The largest domains fill train first, and the remaining domains are shuffled and dealt to validation and test, with seed 0.

`prepare_data.run_items` also checks that every domain sits in one split, that item ids run from 0 to n-1, and that labels match the class folder.

In [4]:
# Build and save the items table and its summary, then read it back the way the model notebooks do
prepare_data.run_items(root)
items = prepare_data.load_items(PROC_DIR)
print(items.shape)
items.head()


saved c:\Users\joetn\convolutional_vae_quishing\data\processed\items.csv.gz: 1,005,700 items, data version 08d6fa626da8
saved c:\Users\joetn\convolutional_vae_quishing\data\processed\items_summary.json
(1005700, 18)


,item_id,cls,label,index,url,split,domain,host,url_len,url_len_bin,truncated,scheme,has_scheme,is_https,host_is_ip,n_host_labels,path_len,digit_ratio
0,0,benign,0,1,http://freshrpms.net/,train,freshrpms.net,freshrpms.net,21,1-32,False,http,True,False,False,2,1,0.000000
1,1,benign,0,2,http://lists.freshrpms.net/mailman/listinfo/rp...,train,freshrpms.net,lists.freshrpms.net,52,33-64,False,http,True,False,False,3,26,0.000000
2,2,benign,0,3,http://www.boquist.net/stort-sup-brev,validation,boquist.net,www.boquist.net,37,33-64,False,http,True,False,False,3,15,0.000000
3,3,benign,0,6,http://hesketh.com,test,hesketh.com,hesketh.com,18,1-32,False,http,True,False,False,2,0,0.000000
4,4,benign,0,7,http://seeker.dice.com/seeker.epl?rel_code=31,validation,dice.com,seeker.dice.com,45,33-64,False,http,True,False,False,3,11,0.044444


## 3. Checks on the splits

Two checks that must hold: no domain appears in more than one split, and no validation or test row shares a host with a train row of its own class. Then the counts per split.

In [5]:
# Checks (each domain sits in one split, no host is shared with train)
checks = {}
for cls, g in items.groupby("cls"):
    train_hosts = set(g.loc[g["split"] == "train", "host"])
    checks[cls] = {
        "domains found in more than one split": int((g.groupby("domain")["split"].nunique() > 1).sum()),
        "validation or test rows sharing a host with train": int(g.loc[g["split"] != "train", "host"].isin(train_hosts).sum()),
    }
display(pd.DataFrame(checks))
assert all(v == 0 for c in checks.values() for v in c.values())

,benign,malicious
domains found in more than one split,0,0
validation or test rows sharing a host with train,0,0


In [6]:
# Class counts per split, the malicious share of each split, and how large the biggest domain is
counts = items.pivot_table(index="split", columns="cls", values="item_id", aggfunc="count").loc[SPLIT_NAMES]
counts["malicious share of split"] = counts["malicious"] / (counts["benign"] + counts["malicious"])
counts["benign share of class"] = counts["benign"] / counts["benign"].sum()
counts["malicious share of class"] = counts["malicious"] / counts["malicious"].sum()
counts["largest domain, share of split rows"] = pd.Series({s: g["domain"].value_counts(normalize=True).iloc[0] for s, g in items.groupby("split")})
counts.round(4)

cls,benign,malicious,malicious share of split,benign share of class,malicious share of class,"largest domain, share of split rows"
split,,,,,,
train,300969,403020,0.5725,0.70,0.70,0.021
validation,64494,86361,0.5725,0.15,0.15,0.000
test,64494,86362,0.5725,0.15,0.15,0.000


**Reading.** Each class lands close to 70/15/15, and every split is about 57% malicious, so the AUPRC of a random scorer is about 0.57 in all three. No domain is split across train, validation and test, and no validation or test row shares a host with a training row of its own class. The largest domain (about 2% of the train rows) sits in train, and the largest domain in validation or test is a negligible share, as intended.

Two things to know about the grouping:

- A host that is an IP address has no registered domain, so the `domain` column holds the last two numbers of the address (for example `115.240`). Rows of the same IP always stay together, and unrelated IPs that share those two numbers move together too.
- The classes are split separately, so a domain that appears in both classes can land in different splits. The next cell counts how often this affects malicious validation and test rows.

In [7]:
# Malicious validation and test rows whose domain also appears in the benign train split
benign_train_domains = set(items.loc[(items["cls"] == "benign") & (items["split"] == "train"), "domain"])
mal_eval = items[(items["cls"] == "malicious") & (items["split"] != "train")]
shared = mal_eval["domain"].isin(benign_train_domains)
print(f"{shared.sum():,} of {len(mal_eval):,} malicious validation and test rows ({shared.mean():.3%}) share a domain with a benign train row")

99 of 172,723 malicious validation and test rows (0.057%) share a domain with a benign train row


## 4. What uses which rows

| Model | Fitted on | Tuned on | Threshold set on | Scored on |
| --- | --- | --- | --- | --- |
| Floors and the VAE | benign train rows | validation rows | benign validation rows | validation and test rows |
| Supervised reference | benign and malicious train rows | validation rows | benign validation rows | validation and test rows |

The test rows are used once per model, after the configuration is fixed on validation. Malicious train rows are used only by the supervised reference.

The model notebook reads the items table with `load_items` and takes these rows from it.

### Files this notebook saved in `data/processed/`

| File | What it holds |
| --- | --- |
| `items.csv.gz` | the items table read by the model notebook |
| `items_summary.json` | counts per split and truncated URLs, and a short `data_version` that changes if the rows or splits change |